# Native six-bit walk reference (catalyst source)

The original Notebook contains only commented code. This runnable migration uses
FlagQuantum exclusively; it does not execute or depend on the original SDK.

Preserve the original loop indentation: X(x[0]) occurs inside all five MCX iterations. This is a reversible permutation, but differs from modular increment on 32 of 64 positions. Do not silently replace it with arithmetic increment.

Target basis: CX/RZ/RY/RX. Exact logical phase uses one final RZ on an existing clean zero work qubit. Its cost is included; the transform is an isometry with zero input/output work, not an identity for dirty work.

The source requests CX/RZ/RY/RX then counts CNOT text in a jitted MLIR string. We count actual executable CX instructions instead. No qjit, Lightning runtime or MLIR-text timing equivalence is claimed.


In [ ]:
from typing import Annotated

import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import (
    H,
    X,
    Output,
    QArray,
    QBit,
    QNum,
    adjoint,
    allocate,
    control,
    qfunc,
)


@qfunc
def cascade_step(index: int, x: QArray[QBit]) -> None:
    control(x[0:index], lambda: X(x[index]))
    X(x[0])


@qfunc
def shift(x: QArray[QBit]) -> None:
    for index in range(5, 0, -1):
        cascade_step(index, x)


@qfunc
def inverse_shift(x: QArray[QBit]) -> None:
    with adjoint():
        shift(x)


@qfunc
def walk(x: Output[QArray[QBit]], coin: Output[QBit]) -> None:
    allocate(6, x)
    allocate(coin)
    H(coin)
    control(coin == 1, lambda: shift(x))
    control(coin == 0, lambda: inverse_shift(x))


program = synth.synthesize(walk, max_qubits=16, max_workspace=9, max_operations=20_000)
result = fq.run(program)
print(dict(program.probabilities(result, registers=("x", "coin"))))


In [ ]:
from examples.synthesis.tutorials.technology_demonstrations.classiq_paper.quantum_walk.reference_workflows import (
    run_workflow,
)

workflow = run_workflow("catalyst")
print(workflow.resources)
print("synthesis_seconds", workflow.synthesis_seconds)
print("lowering_seconds", workflow.lowering_seconds)
print("local_workflow_seconds", workflow.elapsed_seconds)


In [ ]:
from examples.synthesis.tutorials.technology_demonstrations.classiq_paper.quantum_walk.reference_circuit import (
    reference_circuit,
)
from examples.synthesis.tutorials.technology_demonstrations.classiq_paper.quantum_walk.reference_workflows import (
    logical_state,
)
import torch

expected = fq.run(reference_circuit(catalyst=True)).statevector().flatten()
for executable in (program.circuit, workflow.executable):
    torch.testing.assert_close(
        logical_state(executable, program), expected, atol=1e-11, rtol=0
    )
print("independent_full_state_check", "passed")


Source outputs are empty in version control. Tests also check every logical
input column and complex coherent inputs; this initial-state comparison alone
does not prove phase or cleanup. No original compiler timing is reproduced.
